Installing dependencies

In [2]:
%pip install boto3 pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Setting up AWS connection

In [5]:
import boto3
import pandas as pd
import io

s3 = boto3.client('s3')
bucket_name = 'cleaned-filtered-data'
file_key = 'V_Dem_Filtered.csv' 

obj = s3.get_object(Bucket=bucket_name, Key=file_key)
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

Prepare data for model training

In [7]:
df_modern = df[df['year'] >= 1945].copy()
main_score = 'v2x_polyarchy' 

df_modern = df_modern.sort_values(['country_id', 'year'])
df_modern['dem_change_3yr'] = df_modern.groupby('country_id')[main_score].shift(-3) - df_modern[main_score]

df_clean = df_modern.dropna(subset=[main_score, 'dem_change_3yr', 'e_gdppc'])

# Defines backsliding as the worst 5% of democratic declines
bottom_5_percentile = df_clean['dem_change_3yr'].quantile(0.05)
df_clean['target_backslide_3yr'] = (df_clean['dem_change_3yr'] <= bottom_5_percentile).astype(int)

df_clean['target_gdp_3yr'] = df_clean.groupby('country_id')['e_gdppc'].shift(-3)
df_clean = df_clean.dropna(subset=['target_gdp_3yr'])

Train the gdp model

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score

# 1. Define Features (X) and Target (y) 
X_gdp = df_clean.drop(columns=[
    'year', 'country_name', 'country_id', 'country_text_id', 
    'target_gdp_3yr', 'target_backslide_3yr', 'dem_change_3yr',
    'e_gdppc', 'e_gdp', 'e_miinflat', 'e_uds'
])
y_gdp = df_clean['target_gdp_3yr']

# 2. Initialize the model
# max_depth and min_samples_leaf to force the model to generalize
gdp_model = RandomForestRegressor(
    n_estimators=500, 
    max_depth=10, 
    min_samples_leaf=10, 
    random_state=42,
    n_jobs=-1
)

# 3. Evaluate
# Instead of one lucky split, we take the average of 5 different tests
cv_scores = cross_val_score(gdp_model, X_gdp, y_gdp, cv=5)
honest_r2 = cv_scores.mean()

# 4. Final Fit 
gdp_model.fit(X_gdp, y_gdp)

print(f"Honest Average $R^2$ Score (via Cross-Validation): {honest_r2:.2f}")
print(f"Range of scores across folds: {cv_scores}")

print("\n--- VERIFIED RED FLAGS FOR FUTURE GDP ---")
gdp_flags = pd.Series(gdp_model.feature_importances_, index=X_gdp.columns)
print(gdp_flags.sort_values(ascending=False).head(5))

Honest Average $R^2$ Score (via Cross-Validation): 0.53
Range of scores across folds: [0.74153443 0.69944419 0.38805624 0.46963744 0.32983632]

--- VERIFIED RED FLAGS FOR FUTURE GDP ---
v2x_regime       0.269270
v2x_polyarchy    0.256748
e_wbgi_gee       0.213535
v2x_civlib       0.110456
v2x_corr         0.042057
dtype: float64


Train the backsliding model

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_back = df_clean.drop(columns=[
    'year', 'country_name', 'country_id', 'country_text_id', 
    'target_gdp_3yr', 'target_backslide_3yr', 'dem_change_3yr',
    'v2x_polyarchy', 'e_uds', 'v2x_regime'
])
y_back = df_clean['target_backslide_3yr']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_back, y_back, test_size=0.2, random_state=42, stratify=y_back
)
back_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
back_model.fit(X_train_b, y_train_b)

print("\n--- BACKSLIDING PREDICTION REPORT ---")
# This shows how well we caught the '1's (Backsliders)
print(classification_report(y_test_b, back_model.predict(X_test_b)))

print("\n--- TOP RED FLAGS FOR BACKSLIDING ---")
back_flags = pd.Series(back_model.feature_importances_, index=X_back.columns)
print(back_flags.sort_values(ascending=False).head(5))


--- BACKSLIDING PREDICTION REPORT ---
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      2119
           1       0.67      0.13      0.22       104

    accuracy                           0.96      2223
   macro avg       0.81      0.57      0.60      2223
weighted avg       0.95      0.96      0.94      2223


--- TOP RED FLAGS FOR BACKSLIDING ---
v2x_accountability         0.204148
v2x_electoral_integrity    0.160097
v2x_corr                   0.140544
v2x_civlib                 0.119750
e_gdp                      0.106020
dtype: float64


In [11]:
import io

# 1. Generate the Predictions & Risk Scores
df_clean['predicted_gdp_3yr'] = gdp_model.predict(X_gdp)
df_clean['backslide_risk_score'] = back_model.predict_proba(X_back)[:, 1]

# 2. Package for Tableau
tableau_df = df_clean[[
    'year', 'country_name', 'v2x_polyarchy', 'e_gdppc', 
    'predicted_gdp_3yr', 'backslide_risk_score', 'dem_change_3yr'
]]

# 3. Upload to S3
csv_buffer = io.StringIO()
tableau_df.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket='cleaned-filtered-data', 
    Key='V-Dem/V_Dem_Final_Results.csv', 
    Body=csv_buffer.getvalue()
)

print(" Results uploaded to S3!")

 Results uploaded to S3!


In [12]:
import joblib
import os

joblib.dump(gdp_model, 'gdp_predict_model.joblib')
joblib.dump(back_model, 'backslide_risk_model.joblib')

bucket_name = 'cleaned-filtered-data'

s3.upload_file('gdp_predict_model.joblib', bucket_name, 'models/gdp_predict_model.joblib')
s3.upload_file('backslide_risk_model.joblib', bucket_name, 'models/backslide_risk_model.joblib')

print("Model Artifacts successfully uploaded to S3: models/ folder")

Model Artifacts successfully uploaded to S3: models/ folder
